# LLM Setup Guide - Run Large Language Models on Google Colab

**Deploy large language models with memory optimization for Colab free tier**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## Overview

This notebook demonstrates how to set up and run Large Language Models (LLMs) on Google Colab with:
- **Memory optimization** for free tier limitations
- **Quantization** to reduce model size
- **Secure configuration** for privacy
- **Streaming inference** for better UX

---


## Step 1: Configure Security Environment

In [ ]:
import os

# Security configuration
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/.hf_cache"

# Memory optimization for free tier
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

print("\u2705 Security and memory optimization configured!")

## Step 2: Install Dependencies

Install packages for LLM loading with quantization support.

In [ ]:
# Install required packages
!pip install -q transformers accelerate bitsandbytes sentencepiece

print("\u2705 Dependencies installed!")

## Step 3: Load Model with 8-bit Quantization

For free tier, we use 8-bit quantization to reduce memory usage by ~50%.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Model selection - using GPT-2 as example (small, fast)
# For larger models like LLaMA, Mistral, use quantization
model_name = "gpt2-medium"  # 345M parameters - manageable on free tier

print(f"Loading {model_name} with quantization...")

# Check GPU availability
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with 8-bit quantization for memory efficiency
# Note: For this small model, we don't need quantization, but shown for larger models
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print(f"\u2705 Model loaded successfully!")
print(f"   Parameters: {model.num_parameters():,}")

## Step 4: Secure Text Generation Function

In [ ]:
def generate_securely(
    prompt, 
    max_new_tokens=100, 
    temperature=0.7,
    top_p=0.9,
    do_sample=True
):
    """
    Generate text with security-first configuration.
    
    Args:
        prompt: Input text
        max_new_tokens: Maximum new tokens to generate
        temperature: Sampling temperature (higher = more creative)
        top_p: Nucleus sampling parameter
        do_sample: Whether to use sampling (False = greedy)
    """
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():  # Important for security - no gradient tracking
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Clear input tensors
    del inputs
    
    return generated_text

print("\u2705 Secure generation function defined!")

## Step 5: Test Generation

In [ ]:
# Test generation
test_prompt = "Write a short story about a robot learning to be human:"

print(f"Prompt: {test_prompt}\n")
print("Generating...")

result = generate_securely(
    test_prompt, 
    max_new_tokens=150,
    temperature=0.8
)

print(f"Generated:\n{result}")

## Step 6: Memory Cleanup

In [ ]:
import gc

def secure_cleanup():
    """Securely clear memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("\u2705 Memory cleared securely!")

# Call cleanup
secure_cleanup()

---

## Model Recommendations for Free Tier

| Model | Size | Memory | Use Case |
|-------|------|--------|----------|
| **gpt2** | 124M | ~1GB | Fast testing |
| **gpt2-medium** | 345M | ~2GB | Balanced performance |
| **distilgpt2** | 82M | ~500MB | Very fast |
| **microsoft/DialoGPT-medium** | 345M | ~2GB | Conversational |

### For Larger Models (7B+ parameters)

Use 4-bit or 8-bit quantization with bitsandbytes:

```python
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b",
    quantization_config=quantization_config
)
```

---

## Security Best Practices

1. **Never log sensitive data** - Use secure cleanup
2. **Disable telemetry** - `HF_HUB_DISABLE_TELEMETRY=1`
3. **Use GPU memory** - Never let data persist in RAM longer than needed
4. **Clear after use** - Call `secure_cleanup()` after processing
5. **No persistent storage** - Don't save sensitive data to Google Drive

---

**Next**: Try [Fine-tuning Notebook](./fine_tuning.ipynb) for customizing models!

**Found this helpful?** Star the [GitHub repo](https://github.com/unn-Known1/huggingface-colab-secure-setup)!